# RAG Pipeline Demo — CS 763 Final Project

**Indirect Prompt Injection in Retrieval-Augmented Generation Systems**

This notebook demonstrates the baseline RAG pipeline:
1. Check Ollama is running
2. Load configuration
3. Load and inspect the NQ corpus
4. Initialize the pipeline (embed + index)
5. Run a sample query
6. Display retrieved chunks with scores
7. Display the LLM answer
8. Inspect the retrieval log

> **Environment:** Run inside the `rag-security` conda env (Python 3.11).

In [1]:
# Cell 1: Check Ollama is running with llama3.2:3b
from rag.generator import Generator

gen = Generator(model="llama3.2:3b", host="http://localhost:11434")
reachable = gen.is_reachable()
print(f"Ollama reachable: {reachable}")
if not reachable:
    raise RuntimeError("Ollama is not running. Start it: ollama serve")
gen.assert_model_available()
print("llama3.2:3b is available.")

Ollama reachable: True
llama3.2:3b is available.


In [2]:
# Cell 2: Load configuration
from rag.config import load_config

config = load_config("config.toml")
print(f"Seed: {config.seed}")
print(f"Top-k: {config.top_k}")
print(f"Corpus size: {config.corpus_size}")
print(f"Chunk words: {config.chunk_words}, Overlap: {config.overlap_words}")
print(f"Embed model: {config.embed_model}")
print(f"LLM model: {config.llm_model}")
print(f"ChromaDB path: {config.chroma_path}")
print(f"Collection: {config.collection}")

Seed: 42
Top-k: 5
Corpus size: 500
Chunk words: 200, Overlap: 25
Embed model: sentence-transformers/all-MiniLM-L6-v2
LLM model: llama3.2:3b
ChromaDB path: .chroma
Collection: nq_clean_v1


In [4]:
# Cell 3: Load corpus and show stats
from rag.corpus import load_nq_passages, chunk_text

passages = load_nq_passages(n=config.corpus_size, seed=config.seed)
print(f"Loaded {len(passages)} unique passages")
print(f"Sample passage (ID {passages[0].passage_id}):")
print(f"  Query: {passages[0].query}")
print(f"  Text:  {passages[0].text[:200]}...")

# Show chunking stats
all_chunks = []
for p in passages:
    all_chunks.extend(chunk_text(p.text, config.chunk_words, config.overlap_words))
print(f"Total chunks: {len(all_chunks)}")
avg_words = sum(len(c.split()) for c in all_chunks) / len(all_chunks)
print(f"Avg words/chunk: {avg_words:.1f}")

/opt/anaconda3/envs/rag-security/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded 500 unique passages
Sample passage (ID 0):
  Query: who are the basques and where do they live
  Text:  Basques The Basques (/bɑːsks/ or /bæsks/; Basque: euskaldunak [eus̺kaldunak]; Spanish: vascos [ˈbaskos]; French: basques [bask]) are an indigenous ethnic group[5][6][7] characterised by the Basque lan...
Total chunks: 533
Avg words/chunk: 100.8


In [5]:
# Cell 4: Initialize pipeline (embed + index -- may take ~30s on first run)
from rag.pipeline import RAGPipeline

pipeline = RAGPipeline(config)
pipeline.build()  # loads corpus, indexes into ChromaDB, wires generator + logger
print(f"Pipeline initialized. Collection has {pipeline._retriever.count} chunks.")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9493.96it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Upserting to ChromaDB: 100%|██████████| 1/1 [00:00<00:00,  6.90it/s]

Pipeline initialized. Collection has 533 chunks.


In [9]:
# Cell 5: Run a sample query
question = "what is the boiling point of water"
result = pipeline.query(question)
print(f"Query: {question}")
print(f'Number of hits: {len(result["hits"])}')

Query: what is the boiling point of water
Number of hits: 5


In [11]:
# Cell 6: Display retrieved chunks with similarity scores
print(f'Retrieved chunks for: "{question}"')
for hit in result["hits"]:
    print(f'--- Rank {hit["rank"]} | {hit["chunk_id"]} | Score: {hit["score"]:.4f} ---')
    print(hit["text"][:300])
    print()

Retrieved chunks for: "what is the boiling point of water"
--- Rank 1 | chunk_00386 | Score: 0.2270 ---
Fresh water Out of all the water on Earth, saline water in oceans, seas and saline groundwater make up about 97% of it. Only 2.5–2.75% is fresh water, including 1.75–2% frozen in glaciers, ice and snow, 0.5–0.75% as fresh groundwater and soil moisture, and less than 0.01% of it as surface water in l

--- Rank 2 | chunk_00270 | Score: 0.2251 ---
The pot calling the kettle black The earliest appearance of the idiom is in Thomas Shelton's 1620 translation of the Spanish novel Don Quixote. The protagonist is growing increasingly restive under the criticisms of his servant Sancho Panza, of which one is that "You are like what is said that the f

--- Rank 3 | chunk_00336 | Score: 0.2139 ---
Quantum The concept of quantization of radiation was discovered in 1900 by Max Planck, who had been trying to understand the emission of radiation from heated objects, known as black-body radiation. By 

In [12]:
# Cell 7: Display LLM answer
print("=" * 60)
print("LLM ANSWER")
print("=" * 60)
print(result["answer"])

LLM ANSWER
There is no information about the boiling point of water in the provided context. The context discusses various topics such as freshwater distribution, idioms, quantum physics, ice pellets, and a film, but does not mention the boiling point of water.


In [14]:
# Cell 8: Inspect the retrieval log
import json
from pathlib import Path

log_path = Path(config.log_path)
if log_path.exists():
    lines = log_path.read_text(encoding="utf-8").strip().splitlines()
    print(f'Total log entries: {len(lines)}')
    last = json.loads(lines[-1])
    print(f"Last entry:")
    print(f'  Timestamp: {last["timestamp"]}')
    print(f'  Query: {last["query"]}')
    print(f'  Top-k: {last["top_k"]}')
    print(f'  Results: {len(last["results"])} chunks')
else:
    print(f"Log file not found at {log_path}")

Total log entries: 2
Last entry:
  Timestamp: 2026-04-13T00:27:39.544061+00:00
  Query: what is the boiling point of water
  Top-k: 5
  Results: 5 chunks


In [33]:
# Cell 9: Cleanup
pipeline._logger.close()
print("Pipeline closed.")

ValueError: I/O operation on closed file.